In [18]:
#install data locally

import os
os.environ["KAGGLEHUB_CACHE"] = "./data"

import kagglehub

# Download latest version
path = kagglehub.dataset_download("stevezeyuzhang/rsna-pneumonia-detection-challenge")
print("Path to dataset files:", path)

100%|██████████| 15.2G/15.2G [34:27<00:00, 7.88MB/s]  

Extracting files...


Path to dataset files: ./data\datasets\stevezeyuzhang\rsna-pneumonia-detection-challenge\versions\1


In [32]:
# importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
 
import keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.optimizers import Adam
 
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight


In [ ]:
# defining constants
TRAIN_IMG_DIR = "data\datasets\stevezeyuzhang\\rsna-pneumonia-detection-challenge\\versions\\1\\rsna-pneumonia-detection-challenge\stage_2_train_images_png"  
CSV_PATH      = "data\datasets\stevezeyuzhang\\rsna-pneumonia-detection-challenge\\versions\\1\\rsna-pneumonia-detection-challenge\stage_2_train_labels.csv"
IMG_SIZE      = (112, 112)
BATCH_SIZE    = 64
EPOCHS        = 100
LR            = 0.001

In [34]:
df = pd.read_csv(CSV_PATH)
 
# one row per patient as each patient either has pneumonia or not
df = df.drop_duplicates("patientId").reset_index(drop=True)
 
# Filename to match  PNG filename
df["filename"] = df["patientId"] + ".png"
 
# Labels
df["Target"] = df["Target"].astype(str)
 
print(f"Total patients : {len(df)}")
print(f"Normal  (0)    : {(df['Target'] == '0').sum()}")
print(f"Pneumonia (1)  : {(df['Target'] == '1').sum()}")

Total patients : 26684
Normal  (0)    : 20672
Pneumonia (1)  : 6012


In [35]:

# stratify keeps the same class ratio in both splits
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["Target"],
    random_state=42
)
 
print(f"\nTrain size : {len(train_df)}")
print(f"Val size   : {len(val_df)}")



Train size : 21347
Val size   : 5337


In [36]:
# Augmentation only on training data
trdata = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True,
    shear_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
)
 
# Validation: only rescale, no augmentation
tsdata = ImageDataGenerator(rescale=1.0 / 255)
 
traindata = trdata.flow_from_dataframe(
    dataframe=train_df,
    directory=TRAIN_IMG_DIR,
    x_col="filename",
    y_col="Target",
    target_size=IMG_SIZE,
    class_mode="binary",
    batch_size=BATCH_SIZE,
    shuffle=True,
)
 
testdata = tsdata.flow_from_dataframe(
    dataframe=val_df,
    directory=TRAIN_IMG_DIR,
    x_col="filename",
    y_col="Target",
    target_size=IMG_SIZE,
    class_mode="binary",
    batch_size=BATCH_SIZE,
    shuffle=False,
)
 


Found 21347 validated image filenames belonging to 2 classes.
Found 5337 validated image filenames belonging to 2 classes.


In [37]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["Target"].astype(int),
)
class_weights = {0: weights[0], 1: weights[1]}
print(f"\nClass weights: {class_weights}")
 



Class weights: {0: np.float64(0.6454314567333858), 1: np.float64(2.219022869022869)}


In [42]:
model = Sequential()
 
# Block 1
model.add(Conv2D(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), filters=64, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(Conv2D(filters=64, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2), strides=(2, 2)))
 
# Block 2
model.add(Conv2D(filters=128, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(Conv2D(filters=128, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2), strides=(2, 2)))
 
# Block 3
model.add(Conv2D(filters=256, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(Conv2D(filters=256, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(Conv2D(filters=256, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2), strides=(2, 2)))
 
# Block 4
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2), strides=(2, 2)))
 
# Block 5
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(Conv2D(filters=512, kernel_size=(3, 3), padding="same", activation="relu"))
model.add(MaxPool2D(pool_size=(2, 2), strides=(2, 2)))
 
# Classifier head
model.add(Flatten())
model.add(Dense(units=4096, activation="relu"))
model.add(Dropout(0.5))                        # prevent overfitting
model.add(Dense(units=4096, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(units=1, activation="sigmoid")) # binary output
 
model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_39 (Conv2D)              │ (None, 112, 112, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_40 (Conv2D)              │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_41 (Conv2D)              │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_42 (Conv2D)              │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_43 (Conv2D)              │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_44 (Conv2D)              │ (None, 28, 28, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_45 (Conv2D)              │ (None, 28, 28, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_46 (Conv2D)              │ (None, 14, 14, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_47 (Conv2D)              │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_48 (Conv2D)              │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_49 (Conv2D)              │ (None, 7, 7, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_50 (Conv2D)              │ (None, 7, 7, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_51 (Conv2D)              │ (None, 7, 7, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 3, 3, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 4096)           │    18,878,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │         4,097 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,378,561 (192.18 MB)

 Trainable params: 50,378,561 (192.18 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
opt = Adam(learning_rate=LR)
model.compile(
    optimizer=opt,
    loss="binary_crossentropy",   # correct loss for binary classification
    metrics=["accuracy"]
)
 


In [40]:
checkpoint = ModelCheckpoint(
    "vgg16_pneumonia_best.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1,
)
 
early = EarlyStopping(
    monitor="val_accuracy",
    patience=20,
    mode="max",
    verbose=1,
    restore_best_weights=True,
)
 
 


In [41]:
hist = model.fit(
    traindata,
    steps_per_epoch=len(traindata),
    validation_data=testdata,
    validation_steps=len(testdata),
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[checkpoint, early],
)

 


Epoch 1/100


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense_6" is incompatible with the layer: expected axis -1 of input shape to have value 25088, but received input with shape (None, 4608)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 112, 112, 3), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [ ]:
plt.figure(figsize=(12, 4))
 
plt.subplot(1, 2, 1)
plt.plot(hist.history["accuracy"],     label="Train Accuracy")
plt.plot(hist.history["val_accuracy"], label="Val Accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
 
plt.subplot(1, 2, 2)
plt.plot(hist.history["loss"],     label="Train Loss")
plt.plot(hist.history["val_loss"], label="Val Loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
 
plt.tight_layout()
plt.savefig("training_curves.png")
plt.show()


In [ ]:
from keras.preprocessing import image
from keras.models import load_model
 
def predict_xray(img_path, model_path="vgg16_pneumonia_best.h5"):
    saved_model = load_model(model_path)
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = np.expand_dims(np.asarray(img) / 255.0, axis=0)
    prob = saved_model.predict(img_array)[0][0]
    label = "PNEUMONIA" if prob > 0.5 else "NORMAL"
    print(f"Prediction : {label}  (confidence: {prob:.2f})")
    return label, prob